# Assignment 1: Model

**Author:** Lino Kaiser.

In this assignment, we were tasked with the following:

- Draw the plan of a two-zone building.
- Formulate the hypothesis for boundary conditions.
- Choose the types of windows, doors, and walls.
- Draw the thermal circuit:
  - temperature nodes,
  - flow-rate paths,
  - thermal conductances for conduction, convection, long-wave radiation, advection, and P-controllers,
  - sources of temperature and flow rate.
- Number the temperature nodes and the flow-rate branches (starting from 0).
- Calculate the thermal conductances for conduction, convection, long-wave radiation, and advection.
- Calculate the thermal capacities.
- Write down the incidence matrix $A$, the conductance matrix $G$, and the capacity matrix $C$ of the system of differential-algebraic equations (DAE).
- Define the inputs: temperature sources (vector $b$) and flow-rate sources (vector $f$).
- Write in Python the incidence matrix $A$, the conductance matrix $G$, and the capacity matrix $C$ of the system of differential-algebraic equations (DAE).
- Write in Python the vectors of pointers to the temperature sources $b$, flow-rate sources $f$, and outputs $y$.

I chose to draw the plan of a two-zone building that represents and is supposed to model a ZHAW building.

The plan represents a fictional building inspired by a ZHAW building. I tried to make the construction assumptions as accurate as possible for a real building on the ZHAW Technikum campus. I simplified the design considerably to fit the scope of the project and did not consult sources on how those buildings were constructed.

These definitions form the baseline for the rest of the project and might be referenced there later on.

## Layout and dimensions

(Insert hand drawing here.)

All doors are assumed to be closed. The drawing omits wall thickness.

After defining the inspiration and the dimensions, the next step is to clearly define the assumptions and boundary conditions, as well as choose the types of windows, doors, and walls and what they are made of.

As can be seen in my drawing, the window and door placements have already been defined, so let's proceed with summarizing the dimensions chosen above.

General/total dimensions

| | |
| --- | --- |
| Total area | 54 m² |
| Height | 3 m |
| Air volume | 162 m³ |
| Occupants | 6 |
| Windows | 2 |
| External wall thickness | 0.35 m |
| Internal wall thickness | 0.25 m |
| Floor thickness | 0.25 m |
| Roof thickness | 0.35 m |
| Convection coefficient $h_c$ | 3 W/(m²·K) |
| Radiation coefficient $h_r$ | 5 W/(m²·K) |
| Door thickness | 0.04 m |
| Door width | 1 m |
| Door height | 2.1 m |
| U-values for the doors and windows | 2 W/(m²·K) |
| Air density | 1.2 kg/m³ |
| Specific heat capacity | 1000 J/(kg·K) |
| Air change rate | 1 h⁻¹ |

Zone 1, student room only

| | |
| --- | --- |
| Total area | 36 m² |
| Air volume | 108 m³ |
| Occupants | 4 |
| Windows | 1 (south-facing) |
| Window | 4 m × 2 m = 8 m² |
| Room volume | 108 m³ |

Zone 2, office only

| | |
| --- | --- |
| Total area | 18 m² |
| Air volume | 54 m³ |
| Occupants | 2 |
| Windows | 1 (north-facing) |
| Window | 2 m × 2 m = 4 m² |
| Room volume | 54 m³ |

For each material, we need to know the thermal conductivity and the specific heat capacity.

Materials

| Building element | Material | Thermal conductivity [W/(m·K)] | Specific heat capacity [J/(kg·K)] | Density [kg/m³] |
| --- | --- | --- | --- | --- |
| Walls | Solid clay brick | 0.8 | 1000 | 1800 |
| Roof and floor | Concrete | 2 | 1000 | 2400 |
| Doors | Timber | 0.13 | 1600 | 500 |
| Windows | Double glazing | | | |

These are approximate values I assumed based on research using the internet and AI.

The next step is to draw the thermal circuit.

(Insert drawing here)

The next step is to calculate the thermal conductances for conduction, convection, long-wave radiation, and advection.

Let's start by calculating the conductances for conduction. For this, we need the following three parameters:

- Thermal conductivity in W/(m·K), which we defined above
- Cross-sectional area in m², which is easy to calculate
- Material thickness in m, which we defined above

Let's work through an example for the external wall of the student room. The total wall surface area is 54 m², the thermal conductivity is 0.8 W/(m·K), and the thickness is 0.35 m.

Then we have (0.8 W/(m·K) × 54 m²) / 0.35 m ≈ 123.42 W/K. Because we model the wall from the outside to the center and from the center to the inside, each side can be considered to have a thickness of 0.175 m. That gives 246.86 W/K for $G_0$ and $G_1$. The next step is to define this process either in Python or by hand. Considering that we have 32 conductance values, both are reasonable approaches.

I chose to do it in Python. First, I define the conduction formula as a function so it can be reused. Next, let's define all the conductances and let Python calculate the values. I will start with the wall as a proof of concept.

In [8]:
%run conditions.py

def conduction(cond, area, thickness):

    return cond * area / thickness

G0 = conduction(brick["cond"], external_wall["area"], (external_wall["thickness"] / 2 ))
G1 = G15 = G16 = G0

G4 = conduction(concrete["cond"], roof["area"], (roof["thickness"] / 2 ))
G5 = G19 = G20 = G4

G8 = conduction(brick["cond"], floor["area"], (floor["thickness"] / 2 ))
G9 = G23 = G24 = G8

Next, let's calculate convection and radiation conductances in W/K. For this, we need the coefficients $h_c$ and $h_r$, both in W/(m²·K).

In [9]:
def convection (hc, area):
    return hc * area

def radiation (hr, area):
    return hr * area

G2 = convection(hc, external_wall["area"])
G17 = G2

G6 = convection(hc, roof["area"])
G21 = G6

G10 = convection(hc, floor["area"])
G25 = G10



G3 = radiation(hr, external_wall["area"])
G18 = G3

G7 = radiation(hr, roof["area"])
G22 = G7

G11 = radiation(hr, floor["area"])
G26 = G11

Okay, that leaves the windows, doors, ventilation, and internal wall. For the internal wall, we can reuse the same approach as for the external wall.

In [10]:
G29 = conduction(brick["cond"], internal_wall["area"], (internal_wall["thickness"] / 2 ))
G30 = G29

G13 = door["u"] * door["area"]
G31 = G13

G12 = student_window["u"] * student_window["area"]
G27 = office_window["u"] * office_window["area"]

That leaves ventilation. The formula in the source for ventilation conductance is air density × specific heat capacity × ((air change rate × volume) / 3600).

$G_v = \rho_{\mathrm{air}} c_{\mathrm{air}} \dot{V}$ [W/K]

$\dot{V} = \mathrm{ACH} \cdot V_{\mathrm{air}} / 3600$ [m³/s]

- $\rho_{\mathrm{air}}$: air density [kg/m³]
- $c_{\mathrm{air}}$: specific heat capacity of air [J/(kg·K)]
- $V_{\mathrm{air}}$: air volume in the room [m³]
- $\mathrm{ACH}$: air changes per hour [h⁻¹]

The factor of 3600 s/h converts hours to seconds.

In [11]:
G14 = air_density * specific_heat_capacity * ((air_change_rate * student_volume)/3600)
G28 = air_density * specific_heat_capacity * ((air_change_rate * office_volume)/3600)

So far, so good. Next are the capacities. According to the source, the thermal capacities in J/K are calculated using density in kg/m³, specific heat capacity in J/(kg·K), and volume in m³.

In [12]:
C0 = air_density * specific_heat_capacity * student_volume
C1 = brick["density"] * specific_heat_capacity * (external_wall["area"] * external_wall["thickness"])
C5 = C1
C2 = concrete["density"] * specific_heat_capacity * (roof["area"] * roof["thickness"])
C6 = C2
C3 = concrete["density"] * specific_heat_capacity * (floor["area"] * floor["thickness"])
C7 = C3
C8 = brick["density"] * specific_heat_capacity * (internal_wall["area"] * internal_wall["thickness"])
C4 = air_density * specific_heat_capacity * office_volume

The incidence, conductance, and capacity matrices were written by hand in OneNote.

Our next task is to put the matrices and vectors we have into Python. The incidence matrix $A$ is dimensionless and is stored in `conditions.py` because of its size. The conductance matrix $G$ is in W/K and the capacity matrix $C$ is in J/K.

In [13]:
G = np.diag([G0, G1, G2, G3, G4, G5, G6, G7, G8, G9, G10, G11, G12, G13, G14, G15, G16, G17, G18, G19, G20, G21, G22, G23, G24, G25, G26, G27, G28, G29, G30, G31])
C = np.diag([C0, C1, 0, C2, 0, C3, 0, C4, C5, 0, C6, 0, C7, 0, C8])

Next, we need to define the temperature sources and flow sources. Again, OneNote was used.

To convert to state space, we can use the code provided in the source.

We also need the dimensionless indicator vector $y$. Its entries are 1 for output nodes and 0 otherwise.

The output nodes I am interested in are 0 and 7.

In [14]:
b = np.array([1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0])
f = np.array([1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0])
y = np.array([1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0])

**Sources:** Christian Ghiaus, [dm4bem](https://github.com/cghiaus/dm4bem).